In [ ]:
%load_ext autoreload
%autoreload 2

# Otimização Optuna

## Inicialização dos datasets

In [ ]:
from src.Anomaly.Optimizer import AnomalyOptunaOptimizer
from src.Data.Processor import DataStreamProcessor
import pandas as pd
from datetime import datetime

# Definição dos datasets
categorias = ['Consistência']
tamanhos = ['25', '200', '1000']

# categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
# tamanhos = ['25']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

## Todas as características

In [ ]:
id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando otimização para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    optimizer = AnomalyOptunaOptimizer(
        stream=stream,
        n_trials=5,
        discretization_threshold='dinamic',
        target_names=targets,
        n_runs=5
    )
    
    best_params = optimizer.optimize(
        model_name='HST',           # HST, AIF, AE, OIF
        warmup_instances=0,
        experiment_name=nome_experimento,
        num_features=len(features),
        exec_id=id_execucao,
        window_evaluation=100
    )

## Melhores características

In [ ]:
%load_ext autoreload
%autoreload 2

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando otimização para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    optimizer = AnomalyOptunaOptimizer(
        stream=stream,
        n_trials=5,
        discretization_threshold='dinamic',
        target_names=targets,
        n_runs=5
    )
    
    best_params = optimizer.optimize(
        model_name='HST',           # HST, AIF, AE, OIF
        warmup_instances=0,
        experiment_name=nome_experimento,
        num_features=len(features),
        exec_id=id_execucao,
        window_evaluation=100
    )

# Execução do Pipeline

## Inicialização dos Datasets

In [ ]:
from src.Anomaly.Pipeline import AnomalyExperimentRunner
from src.Anomaly.Models import get_anomaly_models
from src.Data.Processor import DataStreamProcessor
import pandas as pd

categorias = ['Consistência', 'Generalização', 'Adaptação', 'Recorrência']
tamanhos = ['25', '200', '1000']

# categorias = ['Generalização', 'Adaptação']
# tamanhos = ['200']

datasets = [f'data/15k/{cat}/{cat}_{tam}.csv' for cat in categorias for tam in tamanhos]

FEATURES = [
    'Max Packet Length',
    'Average Packet Size',
    'Fwd Packet Length Min',
    'Min Packet Length',
    'Fwd Packet Length Max',
    'Packet Length Mean',
    'Fwd Packet Length Mean',
    'Avg Fwd Segment Size',
    'min_seg_size_forward',
    'ACK Flag Count',
    'Flow Duration',
    'Fwd IAT Total',
    'Flow IAT Max',
    'Fwd IAT Max',
    'Flow IAT Std',
    'Fwd IAT Std',
    'Fwd IAT Mean',
    'Flow IAT Mean',
    'Total Length of Fwd Packets',
    'Subflow Fwd Bytes',
    'act_data_pkt_fwd',
    'Subflow Fwd Packets',
    'Total Fwd Packets',
    'Down/Up Ratio',
    'Init_Win_bytes_backward',
    'Total Length of Bwd Packets',
    'Subflow Bwd Bytes',
    'Flow IAT Min',
    'Bwd Packet Length Max',
    'URG Flag Count',
    'Bwd IAT Total',
    'Bwd Packets/s',
    'Init_Win_bytes_forward',
]

## Adaptative Isolation Forest (AIF) - Default

### Todas as características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_aif = {
    'window_size': 256,
    'n_trees': 100,
    'height': None,
    'm_trees': 10,
    'weights': 0.5
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AIF'],
        aif_params=default_aif
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_aif,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Melhores Características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_aif = {
    'window_size': 256,
    'n_trees': 100,
    'height': None,
    'm_trees': 10,
    'weights': 0.5
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AIF'],
        aif_params=default_aif
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_aif,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

## Half Space Trees (HST) - Default

### Todas as Características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_hst = {
    'window_size': 250,
    'number_of_trees': 25,
    'max_depth': 15,
    'anomaly_threshold': 0.50,
    'size_limit': 0.10
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['HST'],
        hst_params=default_hst
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_hst,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

### Melhores características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_hst = {
    'window_size': 250,
    'number_of_trees': 25,
    'max_depth': 15,
    'anomaly_threshold': 0.50,
    'size_limit': 0.10
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['HST'],
        hst_params=default_hst
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.50,
        algorithm_params=default_hst,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

## Autoencoder (AE) - Default

### Todas as características

In [11]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_ae = {
    'hidden_layer': 2,
    'learning_rate': 0.5,
    'threshold': 0.60
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AE'],
        ae_params=default_ae
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.60,
        algorithm_params=default_ae,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload

Iniciando treinamento para: Consistência_25

[Autoencoder] Executando 5 rodada(s) prequencial(is)...

                                                                                          METRICS REPORT | DEFAULT_FULLFEATURES                                                                                          
Algorithm              | F1 (%)            | Prec (%)          | Rec (%)           | MCC               | FP          | FN          | Time (s)        | Discrtz    | Win_Eval   | hidden_layer | learning_rate | threshold
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Autoencoder            |  0.9507 ± 0.0000 |  0.4776 ± 0.0000 | 100.0000 ± 0.0000 |  0.0000 ± 0.0000 | 15003 |    0 | 1.5408 | 0.6000     | 100        | 2  

### Melhores características

In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime

default_ae = {
    'hidden_layer': 2,
    'learning_rate': 0.5,
    'threshold': 0.60
}

id_execucao = datetime.now().strftime("%Y%m%d_%H%M")

for dataset_path in datasets:
    nome_experimento = dataset_path.split('/')[-1].replace('.csv', '')
    print(f"\nIniciando treinamento para: {nome_experimento}")
    
    df = pd.read_csv(dataset_path)
    
    processor = DataStreamProcessor(logging=False, selected_features=FEATURES)
    
    stream, targets, features = processor.create_stream(
        df=df,
        target_label_col='Label',
        binary_label=False,
        normalize_method="MinMaxScaler",
        threshold_var=None,
        threshold_corr=None,
        top_n_features=None,
        return_stream=True,
        extra_ignore_cols=['Source IP', 'Source Port', 'Destination IP', 'Destination Port', 'Protocol', 'Inbound'],
        imputation_method='mediana'
    )
    
    algoritmos = get_anomaly_models(
        stream.get_schema(),
        selected_models=['AE'],
        ae_params=default_ae
    )
    
    runner = AnomalyExperimentRunner(target_names=targets, n_runs=5)
    
    runner.run_anomaly_evaluation(
        stream,
        algorithms=algoritmos,
        window_evaluation=100,
        warmup_instances=0,
        title=nome_experimento,
        discretization=0.60,
        algorithm_params=default_ae,          
        is_optimized=False,                       
        num_features=len(features),
        exec_id=id_execucao
    )